# tbats_jax — GPU panel benchmark on Colab

Fits N independent TBATS models in one `vmap`'d call, on a Colab GPU.

**Important:** each new `N` triggers a fresh XLA compile (~90 s on T4). To get a result in reasonable time, this notebook runs **one decisive `N`** (default 500), then optionally pushes to N=2000 in a second cell. Don't sweep N — the compile cost dominates.

**Setup:** Runtime → Change runtime type → T4 GPU.

**CPU baseline (Apple Silicon M-series, single core):**

| N     | compile | warm wall | per-series |
|-------|---------|-----------|------------|
|   32  |  7.9 s  |   7.1 s   | 223 ms     |
|  100  | 21.4 s  |  20.9 s   | 209 ms     |
|  500  | 99.0 s  |  98.1 s   | 196 ms     |

## 1. Install

In [ ]:
!pip install -q --upgrade "jax[cuda12]" optimistix

## 2. Upload `tbats_jax_colab.zip`

In [ ]:
from google.colab import files
uploaded = files.upload()

import zipfile, os
for name in uploaded:
    if name.endswith('.zip'):
        with zipfile.ZipFile(name) as z:
            z.extractall('/content')
        print('extracted to /content/tbats_jax')

assert os.path.isdir('/content/tbats_jax')

In [ ]:
%cd /content/tbats_jax
!pip install -q -e .

## 3. Verify the GPU backend

In [ ]:
import jax
jax.config.update('jax_enable_x64', True)
print('backend :', jax.default_backend())
print('devices :', jax.devices())
assert jax.default_backend() == 'gpu', 'GPU not active'

## 4. One decisive run at N=500

Expect ~90 s compile + ~30–60 s warm wall on T4. This is the honest headline-comparable number vs the CPU's 98 s warm wall.

In [ ]:
from benchmarks.colab_panel_gpu import run
r_500 = run(N=500, T=1500)

## 5. Push to N=2000 (fits on T4)

Triggers a fresh compile. Extrapolated CPU cost at N=2000: ~390 s. GPU should be the same or faster.

In [ ]:
r_2000 = run(N=2000, T=1500)

## 6. Comparison table

In [ ]:
cpu_per_series_ms = 196  # measured at N=500 on Apple Silicon M-series CPU
print(f'{"N":>6} | {"GPU compile":>12} | {"GPU warm":>10} | {"GPU ms/ser":>12} | {"CPU equiv":>10} | {"speedup":>9}')
print('-' * 78)
for N, r in [(500, r_500), (2000, r_2000)]:
    cpu_eq = N * cpu_per_series_ms / 1000
    speedup = cpu_eq / r['warm_wall']
    print(f'{N:>6} | {r["compile_time"]:>11.1f}s | {r["warm_wall"]:>9.2f}s | {r["per_series_ms"]:>11.2f} | {cpu_eq:>9.1f}s | {speedup:>8.1f}x')